# 📏 파인튜닝 모델 평가 (BLEU / ROUGE / BERTScore + Before·After)

베이스 모델과 파인튜닝 모델을 **같은 평가셋**으로 비교합니다. 결과는 README **IV-2** 표에 들어갑니다.

In [ ]:
# (Colab) 라이브러리 설치
!pip install -q -U \
    transformers==4.46.0 peft==0.13.2 bitsandbytes==0.45.3 \
    datasets==3.0.0 accelerate==1.0.1 \
    evaluate sacrebleu rouge_score bert_score

In [ ]:
import os
os.makedirs('output', exist_ok=True)

import torch, evaluate, pandas as pd
from datasets import load_dataset
from datasets.builder import VerificationMode
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

# ===== 설정 =====
MODEL_ID      = 'Qwen/Qwen2.5-3B-Instruct'
ADAPTER_PATH  = 'DdingDDing0103/koculture-qwen2.5-3b-lora'  # HF Hub repo id
# (HF에 업로드 안 했고 로컬에서 쓸 경우: './output/koculture-lora-final')
N_EVAL        =  500        # 평가 표본 수 (T4에서 500개: 약 1시간 30분). 전체는 약 1,036
MAX_NEW_TOKENS = 128

## 1. 평가셋 로드 (학습과 동일한 seed=42, 9:1 분할의 test)

In [ ]:
ds = load_dataset('huggingface-KREW/KoCulture-Dialogues', split='train',
                  verification_mode=VerificationMode.NO_CHECKS)

def to_chat_format(ex):
    return {'messages': [
        {'role': 'user', 'content': ex['question']},
        {'role': 'assistant', 'content': ex['answer']},
    ]}

ds = ds.map(to_chat_format, remove_columns=ds.column_names)
eval_ds = ds.train_test_split(test_size=0.1, seed=42)['test']

eval_ds = eval_ds.select(range(min(N_EVAL, len(eval_ds))))
prompts    = [ex['messages'][0]['content'] for ex in eval_ds]
references = [ex['messages'][1]['content'] for ex in eval_ds]
print(f'평가 표본 수: {len(prompts)}')

## 2. 모델 로드 (4-bit 베이스 + LoRA 어댑터)

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_config, device_map='auto',
)
model = PeftModel.from_pretrained(base, ADAPTER_PATH)
model.eval()
print('✅ 모델 + 어댑터 로드 완료')

In [ ]:
@torch.no_grad()
def generate(prompt):
    messages = [{'role': 'user', 'content': prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, return_tensors='pt', add_generation_prompt=True,
    ).to(model.device)
    out = model.generate(
        inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,  # 평가는 greedy로 재현성 확보
        pad_token_id=tokenizer.pad_token_id,
    )
    return tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True).strip()

from tqdm.auto import tqdm

# 베이스 모델 예측 (어댑터 비활성화)
base_preds = []
with model.disable_adapter():
    for p in tqdm(prompts, desc='base'):
        base_preds.append(generate(p))

# 파인튜닝 모델 예측 (어댑터 활성화)
ft_preds = []
for p in tqdm(prompts, desc='finetuned'):
    ft_preds.append(generate(p))

## 3. 자동 지표 계산 (BLEU / ROUGE-L / BERTScore)

In [ ]:
bleu   = evaluate.load('sacrebleu')
rouge  = evaluate.load('rouge')
bertsc = evaluate.load('bertscore')

def compute_metrics(preds, refs):
    b = bleu.compute(predictions=preds, references=[[r] for r in refs])['score']
    r = rouge.compute(predictions=preds, references=refs)['rougeL']
    bs = bertsc.compute(predictions=preds, references=refs, lang='ko')
    f1 = sum(bs['f1']) / len(bs['f1'])
    return {'BLEU': round(b, 2), 'ROUGE-L': round(r, 4), 'BERTScore-F1': round(f1, 4)}

m_base = compute_metrics(base_preds, references)
m_ft   = compute_metrics(ft_preds,   references)

result = pd.DataFrame({'베이스 모델': m_base, '파인튜닝 모델': m_ft})
result['개선폭'] = result['파인튜닝 모델'] - result['베이스 모델']
result.to_csv('output/metrics.csv', encoding='utf-8-sig')
print('✅ 저장: output/metrics.csv')
result

## 4. 정성 비교 (Before / After)

학습에 쓰지 않은 신조어 위주의 질문으로 before/after 표를 만듭니다.

In [ ]:
import os
os.makedirs('output', exist_ok=True)

test_questions = [
    '친구가 게임에서 봉산탈춤 추고 있다는데 뭔 뜻이야?',
    '내 추구미는 미니멀한 인테리어인데 어떻게 꾸미면 좋을까?',
    '어제 콘서트 진짜 어마무시했어',
    '쟤 음주운전하다 경찰서 정모 갔대',
    '오늘 발표 폼 미쳤다',
]

rows = []
for q in test_questions:
    with model.disable_adapter():
        before = generate(q)
    after = generate(q)
    rows.append({'질문': q, '파인튜닝 전': before, '파인튜닝 후': after})

comparison = pd.DataFrame(rows)
comparison.to_csv('output/before_after_comparison.csv', index=False, encoding='utf-8-sig')
pd.set_option('display.max_colwidth', None)
print('✅ 저장: output/before_after_comparison.csv')
comparison